In [1]:
# ---- Post-procesado de SNMOT-192: genera versión limpia sin tocar el crudo ----
import json, shutil
import numpy as np
from pathlib import Path
from collections import Counter

REPO_ROOT = Path("../..").resolve()
CRUDO = REPO_ROOT / "data" / "tracking" / "integrado_soccernet_SNMOT-119.json"
POST  = REPO_ROOT / "data" / "tracking" / "integrado_soccernet_SNMOT-119_post.json"

assert CRUDO.exists(), f"No existe el crudo: {CRUDO}"
print(f"Crudo: {CRUDO.name}")

shutil.copy(CRUDO, POST)
with open(POST, encoding="utf-8") as fh:
    data = json.load(fh)
frames = data["frames"]

def en_campo(c):
    return c is not None and -5 <= c[0] <= 110 and -5 <= c[1] <= 73

# Fusionar árbitro (7 IDs -> 1)
arb_ids = set(d["id"] for fr in frames for d in fr if d.get("rol") == "arbitro")
if arb_ids:
    id_arb = int(min(arb_ids))
    for fr in frames:
        for d in fr:
            if d.get("rol") == "arbitro":
                d["id"] = id_arb
    print(f"Árbitro: {len(arb_ids)} IDs -> 1 (ID {id_arb})")

# Filtrar balón falso (saltos > 50 m entre frames consecutivos)
balon_pos = {}
for i, fr in enumerate(frames):
    for d in fr:
        if d.get("rol") == "balon" and en_campo(d.get("campo")):
            balon_pos[i] = d["campo"]
fs = sorted(balon_pos)
falsos = set()
for k in range(1, len(fs)):
    fa, fb = fs[k-1], fs[k]
    if fb - fa > 2:
        continue
    pa, pb = balon_pos[fa], balon_pos[fb]
    if np.hypot(pb[0]-pa[0], pb[1]-pa[1]) > 50:
        falsos.add(fb)
for fi in falsos:
    frames[fi] = [d for d in frames[fi] if d.get("rol") != "balon"]
print(f"Balón falso filtrado: {len(falsos)} frames")

with open(POST, "w", encoding="utf-8") as fh:
    json.dump(data, fh)

roles = Counter(d.get("rol") for fr in frames for d in fr)
print(f"\nGuardado: {POST.name}")
print(f"Roles: {dict(roles)}")
print(f"Crudo intacto: {CRUDO.name}")

Crudo: integrado_soccernet_SNMOT-119.json
Balón falso filtrado: 0 frames

Guardado: integrado_soccernet_SNMOT-119_post.json
Roles: {'jugador': 5505, 'balon': 80}
Crudo intacto: integrado_soccernet_SNMOT-119.json
